# Tugas Pertemuan 12 - Asosiasi Data & Sistem Rekomendasi Dasar

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Program Studi:** Informatika  

## Materi
Association Rule Mining, Market Basket Analysis, Apriori, dan Sistem Rekomendasi Dasar.

Pada praktikum ini, saya membuat dataset transaksi sintetis, mencari frequent itemset menggunakan algoritma Apriori, membentuk aturan asosiasi, lalu membuat rekomendasi produk sederhana menggunakan pendekatan Content-Based Filtering.


## 1. Instalasi dan Import Library

Pada bagian ini saya menyiapkan library yang dibutuhkan. Library utama yang digunakan adalah `pandas`, `numpy`, `matplotlib`, `seaborn`, `mlxtend`, dan `scikit-learn`.


In [ ]:
# Jalankan cell ini di Google Colab.
# Jika mlxtend belum tersedia, cell ini akan memasangnya terlebih dahulu.
try:
    import mlxtend
except ImportError:
    !pip -q install mlxtend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style="whitegrid")
np.random.seed(42)

pd.set_option("display.max_colwidth", 120)


## 2. Generate dan Eksplorasi Dataset Transaksi

Dataset transaksi dibuat secara sintetis berisi 50 transaksi belanja. Setiap transaksi berisi 2 sampai 5 produk. Selain itu, saya menambahkan pola tersembunyi agar beberapa produk sering muncul bersama, misalnya `Roti` dengan `Selai`, dan `Kopi` dengan `Gula`.


In [ ]:
produk = [
    "Roti", "Selai", "Susu", "Sereal", "Telur",
    "Keju", "Kopi", "Gula", "Teh", "Mentega"
]

# Membuat 50 transaksi, setiap transaksi berisi 2 sampai 5 produk
transaksi = []
for i in range(50):
    n_item = np.random.randint(2, 6)
    item = list(np.random.choice(produk, n_item, replace=False))
    transaksi.append(item)

# Menambahkan pola tersembunyi: Roti sering bersama Selai
for i in range(0, 20):
    if "Roti" not in transaksi[i]:
        transaksi[i].append("Roti")
    if "Selai" not in transaksi[i]:
        transaksi[i].append("Selai")

# Menambahkan pola lain: Kopi sering bersama Gula
for i in range(20, 35):
    if "Kopi" not in transaksi[i]:
        transaksi[i].append("Kopi")
    if "Gula" not in transaksi[i]:
        transaksi[i].append("Gula")

df_transaksi = pd.DataFrame({
    "transaksi_id": [f"T{i+1:03d}" for i in range(len(transaksi))],
    "daftar_item": transaksi
})

print("Contoh transaksi:")
display(df_transaksi.head(10))

print("Jumlah transaksi:", len(transaksi))
print("Jumlah produk unik:", len(produk))


**Interpretasi:**

Dataset transaksi berisi daftar produk yang dibeli pelanggan dalam satu transaksi. Pola buatan seperti `Roti` dengan `Selai` dan `Kopi` dengan `Gula` digunakan agar algoritma Apriori dapat menemukan aturan asosiasi yang cukup jelas.


## 3. Frekuensi Kemunculan Produk

Sebelum melakukan Apriori, saya melihat frekuensi kemunculan tiap produk. Langkah ini membantu memahami produk mana yang paling sering dibeli.


In [ ]:
from collections import Counter

counter_produk = Counter()
for item_list in transaksi:
    counter_produk.update(item_list)

freq_produk = pd.DataFrame(counter_produk.items(), columns=["produk", "frekuensi"])
freq_produk = freq_produk.sort_values("frekuensi", ascending=False)

display(freq_produk)

plt.figure(figsize=(10, 5))
sns.barplot(data=freq_produk, x="frekuensi", y="produk")
plt.title("Frekuensi Kemunculan Produk dalam Transaksi")
plt.xlabel("Frekuensi")
plt.ylabel("Produk")
plt.show()


**Interpretasi:**

Produk dengan frekuensi tinggi menunjukkan produk yang sering muncul dalam transaksi. Namun frekuensi saja belum cukup untuk menyatakan hubungan antarproduk. Oleh karena itu, diperlukan association rule mining untuk melihat kombinasi produk yang sering dibeli bersama.


## 4. One-Hot Encoding Transaksi

Agar transaksi dapat diproses oleh algoritma Apriori, daftar item harus diubah menjadi format one-hot encoding. Nilai `True` berarti produk muncul dalam transaksi, sedangkan `False` berarti produk tidak muncul.


In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

print("Data transaksi setelah One-Hot Encoding:")
display(df_onehot.head())

print("Shape one-hot table:", df_onehot.shape)


**Interpretasi:**

Tabel one-hot encoding memudahkan algoritma Apriori untuk menghitung support dari setiap itemset. Setiap kolom mewakili produk, sedangkan setiap baris mewakili satu transaksi.


## 5. Frequent Itemset dengan Apriori

Pada bagian ini saya menjalankan algoritma Apriori dengan beberapa nilai `min_support`. Tujuannya untuk melihat pengaruh nilai minimum support terhadap jumlah itemset yang ditemukan.


In [ ]:
support_results = []

for ms in [0.05, 0.10, 0.20, 0.30]:
    freq = apriori(df_onehot, min_support=ms, use_colnames=True)
    support_results.append({
        "min_support": ms,
        "jumlah_itemset": len(freq)
    })

support_df = pd.DataFrame(support_results)
display(support_df)

plt.figure(figsize=(7, 4))
sns.lineplot(data=support_df, x="min_support", y="jumlah_itemset", marker="o")
plt.title("Pengaruh Min Support terhadap Jumlah Frequent Itemset")
plt.xlabel("Min Support")
plt.ylabel("Jumlah Itemset")
plt.show()

# Menggunakan min_support 0.10 karena biasanya masih menghasilkan jumlah itemset yang cukup wajar
freq_items = apriori(df_onehot, min_support=0.10, use_colnames=True)
freq_items = freq_items.sort_values("support", ascending=False)

print("Top 10 Frequent Itemset:")
display(freq_items.head(10))


**Interpretasi:**

Semakin tinggi nilai `min_support`, semakin sedikit frequent itemset yang ditemukan. Hal ini terjadi karena aturan menjadi lebih ketat. Pada praktikum ini, `min_support = 0.10` dipilih karena masih menghasilkan itemset yang cukup banyak untuk dianalisis.


## 6. Membentuk dan Menyaring Association Rules

Setelah frequent itemset ditemukan, langkah berikutnya adalah membentuk aturan asosiasi. Aturan disaring menggunakan `confidence` dan `lift`. Aturan dengan `lift > 1` dianggap memiliki hubungan positif karena item pada antecedent dan consequent muncul bersama lebih sering dibandingkan jika terjadi secara acak.


In [ ]:
rules = association_rules(freq_items, metric="confidence", min_threshold=0.50)
rules = rules[rules["lift"] > 1].copy()
rules = rules.sort_values(["lift", "confidence"], ascending=False)

# Merapikan kolom antecedents dan consequents agar mudah dibaca
rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ", ".join(list(x)))
rules["consequents_str"] = rules["consequents"].apply(lambda x: ", ".join(list(x)))

cols_show = [
    "antecedents_str", "consequents_str",
    "support", "confidence", "lift"
]

print("Top 10 Association Rules berdasarkan Lift:")
display(rules[cols_show].head(10).round(3))


**Interpretasi:**

Aturan dengan nilai `lift` tertinggi dianggap paling kuat karena menunjukkan hubungan antarproduk yang lebih besar dibandingkan kondisi acak. Jika muncul aturan seperti `Roti -> Selai`, maka aturan tersebut masuk akal secara bisnis karena kedua produk tersebut memang sering digunakan bersama.


## 7. Visualisasi Top Association Rules

Visualisasi ini digunakan untuk melihat aturan asosiasi teratas berdasarkan nilai lift dan confidence.


In [ ]:
top_rules = rules.head(10).copy()
top_rules["rule"] = top_rules["antecedents_str"] + " -> " + top_rules["consequents_str"]

plt.figure(figsize=(10, 6))
sns.barplot(data=top_rules, x="lift", y="rule")
plt.title("Top 10 Association Rules berdasarkan Lift")
plt.xlabel("Lift")
plt.ylabel("Rule")
plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=rules,
    x="support",
    y="confidence",
    size="lift",
    hue="lift",
    sizes=(60, 300),
    palette="viridis"
)
plt.title("Support vs Confidence pada Association Rules")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.show()


**Interpretasi:**

Bar chart menunjukkan aturan mana yang memiliki hubungan paling kuat berdasarkan lift. Scatter plot memperlihatkan hubungan antara support dan confidence. Aturan yang baik biasanya memiliki confidence cukup tinggi dan lift lebih dari 1.


## 8. Katalog Produk untuk Content-Based Filtering

Selain Association Rules, saya juga membuat sistem rekomendasi sederhana berbasis Content-Based Filtering. Pendekatan ini menggunakan kesamaan atribut produk seperti kategori dan harga.


In [ ]:
katalog = pd.DataFrame({
    "produk": produk,
    "kategori": [
        "Bakery", "Bakery", "Dairy", "Bakery", "Dairy",
        "Dairy", "Minuman", "Bumbu", "Minuman", "Dairy"
    ],
    "harga": [
        15000, 18000, 12000, 25000, 24000,
        30000, 22000, 14000, 16000, 28000
    ]
})

display(katalog)


**Interpretasi:**

Katalog produk berisi nama produk, kategori, dan harga. Atribut ini digunakan untuk menghitung kemiripan antarproduk. Produk dengan kategori yang sama dan harga yang relatif dekat akan dianggap lebih mirip.


## 9. Rekomendasi Produk dengan Content-Based Filtering

Pada bagian ini, saya membuat fitur produk dari kategori dan harga. Kategori diubah menjadi one-hot encoding, sedangkan harga dinormalisasi agar skala nilainya tidak terlalu besar. Setelah itu, cosine similarity digunakan untuk menghitung kemiripan antarproduk.


In [ ]:
# One-hot encoding kategori
fitur_kategori = pd.get_dummies(katalog["kategori"], prefix="kategori")

# Scaling harga
scaler = MinMaxScaler()
fitur_harga = pd.DataFrame(
    scaler.fit_transform(katalog[["harga"]]),
    columns=["harga_scaled"]
)

fitur_produk = pd.concat([fitur_kategori, fitur_harga], axis=1)

sim_matrix = cosine_similarity(fitur_produk)

sim_df = pd.DataFrame(
    sim_matrix,
    index=katalog["produk"],
    columns=katalog["produk"]
)

print("Matriks cosine similarity:")
display(sim_df.round(3))


In [ ]:
def rekomendasi_serupa(nama_produk, top_n=3):
    if nama_produk not in katalog["produk"].values:
        raise ValueError(f"Produk {nama_produk} tidak ada di katalog.")

    idx = katalog.index[katalog["produk"] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]

    hasil = katalog.iloc[[i for i, _ in skor]].copy()
    hasil["similarity_score"] = [score for _, score in skor]
    return hasil[["produk", "kategori", "harga", "similarity_score"]]

print("Rekomendasi produk yang mirip dengan Roti:")
display(rekomendasi_serupa("Roti", top_n=3).round(3))


**Interpretasi:**

Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan atribut produk. Jika produk target adalah `Roti`, maka produk yang direkomendasikan biasanya berasal dari kategori yang sama atau memiliki harga yang relatif dekat.


## 10. Membandingkan Association Rules dan Content-Based Filtering

Pada bagian ini saya membandingkan rekomendasi dari Association Rules dan Content-Based Filtering untuk produk target yang sama, yaitu `Roti`.


In [ ]:
produk_target = "Roti"

# Rekomendasi dari Association Rules
rules_terkait = rules[rules["antecedents"].apply(lambda x: produk_target in x)].copy()
rules_terkait = rules_terkait.sort_values("lift", ascending=False)

print("Rekomendasi dari Association Rules untuk produk:", produk_target)
if len(rules_terkait) > 0:
    display(rules_terkait[["consequents_str", "support", "confidence", "lift"]].head(5).round(3))
else:
    print("Tidak ada aturan terkait produk target.")

print("
Rekomendasi dari Content-Based Filtering untuk produk:", produk_target)
display(rekomendasi_serupa(produk_target, top_n=3).round(3))


**Interpretasi Perbandingan:**

Association Rules memberikan rekomendasi berdasarkan pola pembelian pelanggan. Jika banyak pelanggan membeli `Roti` bersama `Selai`, maka `Selai` akan direkomendasikan.

Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan atribut produk. Jika `Roti` berada pada kategori `Bakery`, maka produk dengan kategori yang sama seperti `Selai` atau `Sereal` dapat direkomendasikan.

Kedua pendekatan dapat menghasilkan rekomendasi yang mirip, tetapi dasar pengambilan keputusannya berbeda. Association Rules berbasis perilaku transaksi, sedangkan Content-Based Filtering berbasis atribut produk.


## 11. Evaluasi Sederhana dengan Precision@K dan Recall@K

Evaluasi sederhana dilakukan dengan asumsi produk yang relevan untuk `Roti` adalah `Selai`, `Sereal`, dan `Mentega`. Nilai ini hanya contoh untuk menunjukkan cara menghitung Precision@K dan Recall@K.


In [ ]:
def precision_at_k(rekomendasi, item_relevan, k=3):
    top_k = rekomendasi[:k]
    relevan_ditemukan = len(set(top_k) & set(item_relevan))
    return relevan_ditemukan / k

def recall_at_k(rekomendasi, item_relevan, k=3):
    top_k = rekomendasi[:k]
    relevan_ditemukan = len(set(top_k) & set(item_relevan))
    return relevan_ditemukan / len(item_relevan)

item_relevan_roti = ["Selai", "Sereal", "Mentega"]

# Rekomendasi association rules
if len(rules_terkait) > 0:
    rekom_ar = []
    for consequents in rules_terkait["consequents"]:
        rekom_ar.extend(list(consequents))
    rekom_ar = list(dict.fromkeys(rekom_ar))
else:
    rekom_ar = []

# Rekomendasi content-based
rekom_cb = rekomendasi_serupa("Roti", top_n=3)["produk"].tolist()

evaluasi = pd.DataFrame({
    "Metode": ["Association Rules", "Content-Based Filtering"],
    "Rekomendasi": [rekom_ar[:3], rekom_cb],
    "Precision@3": [
        precision_at_k(rekom_ar, item_relevan_roti, k=3) if len(rekom_ar) >= 3 else np.nan,
        precision_at_k(rekom_cb, item_relevan_roti, k=3)
    ],
    "Recall@3": [
        recall_at_k(rekom_ar, item_relevan_roti, k=3) if len(rekom_ar) >= 3 else np.nan,
        recall_at_k(rekom_cb, item_relevan_roti, k=3)
    ]
})

display(evaluasi)


**Interpretasi Evaluasi:**

Precision@K menunjukkan berapa banyak rekomendasi teratas yang relevan. Recall@K menunjukkan seberapa banyak item relevan yang berhasil ditemukan dari seluruh item relevan yang tersedia. Pada contoh ini, evaluasi masih sederhana karena item relevan ditentukan secara manual.


## 12. Kesimpulan

Pada praktikum Pertemuan 12 ini, saya mempelajari dua pendekatan rekomendasi dasar, yaitu Association Rule Mining dan Content-Based Filtering. Association Rule Mining menggunakan pola transaksi untuk menemukan hubungan antarproduk, sedangkan Content-Based Filtering menggunakan kemiripan atribut produk.

Algoritma Apriori membantu menemukan frequent itemset dan membentuk aturan seperti `jika membeli A, maka cenderung membeli B`. Metrik penting yang digunakan adalah support, confidence, dan lift. Nilai lift lebih dari 1 menunjukkan hubungan positif antarproduk.

Dari perbandingan yang dilakukan, Association Rules lebih cocok digunakan ketika data transaksi pelanggan tersedia, sedangkan Content-Based Filtering lebih cocok digunakan ketika informasi atribut produk cukup lengkap. Dalam praktik nyata, kedua pendekatan ini dapat digabungkan menjadi sistem rekomendasi hybrid agar hasil rekomendasi lebih kuat.
